# angel-ai on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ianktoo/angel-ai/blob/main/notebooks/colab_train.ipynb)

Runs the angel-ai finetuning/optimization/evaluation pipeline on a Colab GPU via `backend=cuda` -- the same config path any local NVIDIA box uses (see [`docs/colab.md`](https://github.com/ianktoo/angel-ai/blob/main/docs/colab.md) for the full writeup).

**Before running:** in the Colab menu, go to `Runtime > Change runtime type` and select a GPU (T4 is fine for the defaults below).

This is a research/experimentation pipeline (see the repo README) -- treat these steps as a verified starting point, not a guarantee across every Colab runtime version.

## 1. Clone the repo and bootstrap the environment

In [ ]:
!git clone https://github.com/ianktoo/angel-ai angel-ai
%cd angel-ai

In [ ]:
# Installs uv and runs `uv sync --extra cuda`.
!bash scripts/colab_bootstrap.sh

In [ ]:
# Mount Google Drive so checkpoints/MLflow data survive a session reset.
# This MUST run directly in a notebook cell (not via !bash) -- drive.mount()
# talks to the Colab frontend through this notebook's own IPython kernel,
# which doesn't exist inside a subprocess spawned by a shell script.
from google.colab import drive

drive.mount("/content/drive")

## 2. Configure the run

`output_dir` is set under Google Drive so checkpoints and MLflow data survive a session reset -- use the **same value** in every cell below.

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/angel-ai-runs/run-001"
MODEL = "qwen2_5_0_5b"  # or qwen2_5_1_5b, or point at any HF model repo via model.name=...
DATA = "tiny"  # bundled 4-record example; try data=hf_alpaca for a real Hub dataset
TRAINING = "qlora"  # QLoRA needs backend=cuda (bitsandbytes) -- this is the natural place to run it

## 3. Train

In [ ]:
!uv run python -m angel_ai.training \
  backend=cuda \
  model={MODEL} \
  data={DATA} \
  training={TRAINING} \
  output_dir={OUTPUT_DIR}

### Resuming after a session reset

Re-run the clone, bootstrap, and Drive-mount cells above first (the Colab VM and its filesystem are wiped on each new session -- only Drive survives), then:

In [ ]:
!uv run python -m angel_ai.training \
  backend=cuda \
  model={MODEL} \
  data={DATA} \
  training={TRAINING} \
  output_dir={OUTPUT_DIR} \
  training.resume_from_checkpoint=latest

## 4. Evaluate

If your dataset has no `validation` split (e.g. `data=hf_alpaca`), set `eval.max_eval_samples` so this doesn't evaluate an entire multi-thousand-row dataset.

In [ ]:
!uv run python -m angel_ai.evaluation \
  backend=cuda \
  model={MODEL} \
  data={DATA} \
  output_dir={OUTPUT_DIR} \
  eval.max_eval_samples=20

## 5. Export (optional)

Exports + quantizes the checkpoint to ONNX. `optimize.target=cpu` is the safest choice off this machine's specific hardware (`npu`/`igpu` targets are tied to the Ryzen AI iGPU/NPU this repo was developed against).

In [ ]:
!uv run python -m angel_ai.optimize.export \
  backend=cuda \
  model={MODEL} \
  training={TRAINING} \
  optimize.target=cpu \
  output_dir={OUTPUT_DIR}

## 6. Viewing MLflow results

Colab doesn't conveniently expose a long-running UI port, so the simplest path is to copy the Drive-persisted `mlruns/` folder down locally and run `mlflow ui` there:

```bash
# on your local machine, after syncing the Drive folder down:
mlflow ui --backend-store-uri sqlite:///<output_dir>/mlruns/mlflow.db
```

Alternatively, query metrics directly in this notebook:

In [ ]:
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{OUTPUT_DIR}/mlruns/mlflow.db")
runs = mlflow.search_runs(experiment_names=["angel-ai-training", "angel-ai-eval"])
runs